# NB04 — Hyperparameter Tuning HDBSCAN

**Tujuan:** Menemukan kombinasi parameter optimal untuk pipeline clustering NB02.

**Strategi:**
- Build dense distance matrix **sekali per** `k_neighbors` — reuse untuk semua `min_cluster_size` × `min_samples`
- Metrik evaluasi: `n_clusters`, `coverage_pct`, `relative_validity_` (DBCV approx dari clusterer)

**Input:** `output_nb01/embeddings.npy`  
**Output:** `output_nb04/tuning_results.pkl`, `output_nb04/tuning_results.csv`, plot heatmap

---

> **Target:** `n_clusters < 150` (dataset punya < 150 orang), coverage ≥ 80%


## 0. Instalasi & Import

In [ ]:
import subprocess, sys

def install_faiss():
    for pkg in ["faiss-gpu-cu12", "faiss-gpu-cu11", "faiss-cpu"]:
        result = subprocess.run(
            [sys.executable, "-m", "pip", "install", pkg, "-q"],
            capture_output=True, text=True
        )
        if result.returncode == 0:
            print(f"✅ Berhasil install: {pkg}")
            return pkg
        if result.stderr:
            print(f"  ↳ {pkg} gagal: {result.stderr[:200]}")
    raise RuntimeError("Gagal install faiss — coba manual: pip install faiss-cpu")

try:
    import faiss
    print(f"✅ FAISS sudah ada: {faiss.__version__}")
except ImportError:
    install_faiss()

subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "hdbscan", "scikit-learn", "matplotlib", "pandas", "scipy", "-q"],
    check=True
)


In [ ]:
import pickle
import time
import warnings
from pathlib import Path

import faiss
import hdbscan
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

warnings.filterwarnings("ignore")

print(f"faiss   : {faiss.__version__}")
print(f"hdbscan : {hdbscan.__version__}")

try:
    ngpu = faiss.get_num_gpus()
    print(f"FAISS GPU: {ngpu}  {chr(10)}  {'✅ aktif' if ngpu > 0 else '⚠️  CPU only'}")
except Exception:
    print("FAISS GPU: tidak tersedia")


## 1. Konfigurasi

In [ ]:
# Jika pakai Google Drive — jalankan cell ini
from google.colab import drive
drive.mount("/content/drive", force_remount=True)


In [ ]:
# ─── SESUAIKAN PATH INI ───────────────────────────────────────────────────────
INPUT_EMB_DIR = Path("/content/drive/MyDrive/OTW S.KOM/Embeddings/output_nb01")
OUTPUT_DIR    = Path("/content/drive/MyDrive/OTW S.KOM/Embeddings/output_nb04")
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ─── Parameter Grid ───────────────────────────────────────────────────────────
K_NEIGHBORS_GRID      = [10, 15, 20, 25]
MIN_CLUSTER_SIZE_GRID = [10, 15, 20, 25, 30, 40, 50]
MIN_SAMPLES_GRID      = [3, 5, 10, 15]

TARGET_MAX_CLUSTERS   = 150   # target: n_clusters < jumlah orang asli
MIN_COVERAGE          = 80.0  # coverage minimum yang diterima (%)

K_MAX = max(K_NEIGHBORS_GRID)

total_combinations = len(K_NEIGHBORS_GRID) * len(MIN_CLUSTER_SIZE_GRID) * len(MIN_SAMPLES_GRID)

print(f"Input embeddings : {INPUT_EMB_DIR}")
print(f"Output dir       : {OUTPUT_DIR}")
print()
print(f"Parameter grid:")
print(f"  k_neighbors      : {K_NEIGHBORS_GRID}")
print(f"  min_cluster_size : {MIN_CLUSTER_SIZE_GRID}")
print(f"  min_samples      : {MIN_SAMPLES_GRID}")
print(f"  Total kombinasi  : {total_combinations}")
print()
print(f"Target: n_clusters < {TARGET_MAX_CLUSTERS},  coverage >= {MIN_COVERAGE}%")


## 2. Load Data

In [ ]:
embeddings = np.load(INPUT_EMB_DIR / "embeddings.npy").astype(np.float32)
N, D = embeddings.shape

print(f"Embeddings shape : {embeddings.shape}  dtype={embeddings.dtype}")
print(f"Estimasi dense matrix : {N}×{N}×8 bytes = {N*N*8/1024**3:.2f} GB")


## 3. FAISS — One-Time Search (k = K_MAX)

FAISS index dibangun sekali. Search dilakukan dengan `k = K_MAX` — untuk k lebih kecil
cukup ambil `dist_all[:, :k]` tanpa search ulang.


In [ ]:
t0 = time.time()

emb_f32 = embeddings.copy()
faiss.normalize_L2(emb_f32)

# ── Build index ───────────────────────────────────────────────────────────────
try:
    res   = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, faiss.IndexFlatIP(D))
    print(f"FAISS GPU index ({D} dim)...")
except Exception:
    index = faiss.IndexFlatIP(D)
    print(f"FAISS CPU index ({D} dim)...")

index.add(emb_f32)

# ── Search dengan K_MAX+1 (termasuk self) ─────────────────────────────────────
sim_all, idx_all = index.search(emb_f32, K_MAX + 1)

# Hapus self-match (kolom pertama)
sim_all = sim_all[:, 1:]                               # (N, K_MAX)
idx_all = idx_all[:, 1:]                               # (N, K_MAX)
dist_all = np.clip(1.0 - sim_all, 0.0, 2.0).astype(np.float32)  # cosine distance

t_faiss = time.time() - t0
print(f"FAISS selesai ({t_faiss:.1f} detik)")
print(f"dist_all shape : {dist_all.shape}  — reuse untuk semua k ≤ {K_MAX}")


## 4. Grid Search

Untuk setiap `k_neighbors`: build sparse → dense matrix sekali,  
lalu sweep semua `min_cluster_size` × `min_samples` pada matrix yang sama.


In [ ]:
results = []
total = total_combinations
done  = 0

print(f"Grid search: {total} kombinasi")
print(f"{'─'*75}")
t_start = time.time()

for k in K_NEIGHBORS_GRID:
    # ── Build sparse matrix untuk k ini ──────────────────────────────────────
    dist_k = dist_all[:, :k].astype(np.float32)
    idx_k  = idx_all[:, :k]

    rows = np.repeat(np.arange(N), k)
    cols = idx_k.ravel()
    data = dist_k.ravel()
    sparse = csr_matrix((data, (rows, cols)), shape=(N, N))
    sparse = sparse.maximum(sparse.T)   # simetrisasi

    # ── Build dense float64 matrix ────────────────────────────────────────────
    dense = np.full((N, N), 2.0, dtype=np.float64)
    np.fill_diagonal(dense, 0.0)
    cx = sparse.tocoo()
    dense[cx.row, cx.col] = cx.data.astype(np.float64)

    print(f"k={k}: dense matrix siap ({dense.nbytes/1024**3:.2f} GB)")

    # ── Sweep mcs × ms ────────────────────────────────────────────────────────
    for mcs in MIN_CLUSTER_SIZE_GRID:
        for ms in MIN_SAMPLES_GRID:
            t0 = time.time()
            done += 1

            try:
                clusterer = hdbscan.HDBSCAN(
                    min_cluster_size=mcs,
                    min_samples=ms,
                    metric="precomputed",
                    cluster_selection_method="eom",
                    n_jobs=-1
                )
                clusterer.fit(dense)
                labels = clusterer.labels_

                n_clusters  = int(len(set(labels[labels >= 0])))
                coverage    = float((labels >= 0).sum() / N * 100)
                noise_pct   = float((labels == -1).sum() / N * 100)
                dbcv_approx = float(clusterer.relative_validity_)

            except Exception as e:
                n_clusters  = -1
                coverage    = -1.0
                noise_pct   = -1.0
                dbcv_approx = float("nan")

            t_run = time.time() - t0
            results.append({
                "k_neighbors"      : k,
                "min_cluster_size" : mcs,
                "min_samples"      : ms,
                "n_clusters"       : n_clusters,
                "coverage_pct"     : round(coverage, 2),
                "noise_pct"        : round(noise_pct, 2),
                "dbcv_approx"      : round(dbcv_approx, 4),
                "runtime_s"        : round(t_run, 1),
            })

            t_elapsed = time.time() - t_start
            t_est     = t_elapsed / done * (total - done)
            print(f"  [{done:3}/{total}] mcs={mcs:2} ms={ms:2} → "
                  f"n={n_clusters:3}, cov={coverage:5.1f}%, "
                  f"DBCV={dbcv_approx:+.3f}, {t_run:.0f}s  "
                  f"(sisa ~{t_est/60:.0f} mnt)")

    print()

df = pd.DataFrame(results)
t_total = time.time() - t_start
print(f"Selesai dalam {t_total:.0f} detik ({t_total/60:.1f} menit)")


## 5. Hasil & Analisis

In [ ]:
df_valid = df[df["n_clusters"] > 0].copy()

print(f"Total kombinasi valid   : {len(df_valid)}/{len(df)}")
print(f"Range n_clusters        : {df_valid.n_clusters.min()} – {df_valid.n_clusters.max()}")
print(f"Range coverage          : {df_valid.coverage_pct.min():.1f}% – {df_valid.coverage_pct.max():.1f}%")
print(f"Range DBCV (approx)     : {df_valid.dbcv_approx.min():.3f} – {df_valid.dbcv_approx.max():.3f}")
print()

# Filter sesuai target
df_target = df_valid[
    (df_valid["n_clusters"] < TARGET_MAX_CLUSTERS) &
    (df_valid["coverage_pct"] >= MIN_COVERAGE)
].sort_values("dbcv_approx", ascending=False)

print(f"Kombinasi memenuhi target (n<{TARGET_MAX_CLUSTERS}, cov≥{MIN_COVERAGE}%) : {len(df_target)}")
print()
print("=== TOP 20 (filter target, sort DBCV) ===")
cols_show = ["k_neighbors","min_cluster_size","min_samples",
             "n_clusters","coverage_pct","noise_pct","dbcv_approx"]
print(df_target[cols_show].head(20).to_string(index=False))


In [ ]:
# ── Heatmap: n_clusters per (mcs, ms) untuk setiap k ─────────────────────────
fig, axes = plt.subplots(2, len(K_NEIGHBORS_GRID), figsize=(5*len(K_NEIGHBORS_GRID), 9))
fig.suptitle("Grid Search — n_clusters dan DBCV per parameter",
             fontsize=13, fontweight="bold")

for col, k in enumerate(K_NEIGHBORS_GRID):
    df_k = df_valid[df_valid["k_neighbors"] == k]

    for row, (metric, label, cmap, fmt) in enumerate([
        ("n_clusters",  f"n_clusters (k={k})", "RdYlGn_r", ".0f"),
        ("dbcv_approx", f"DBCV approx (k={k})", "RdYlGn",   ".2f"),
    ]):
        ax = axes[row, col]
        pivot = df_k.pivot(index="min_cluster_size", columns="min_samples", values=metric)

        im = ax.imshow(pivot.values, cmap=cmap, aspect="auto")
        plt.colorbar(im, ax=ax, shrink=0.8)

        ax.set_xticks(range(len(pivot.columns)))
        ax.set_yticks(range(len(pivot.index)))
        ax.set_xticklabels(pivot.columns, fontsize=9)
        ax.set_yticklabels(pivot.index, fontsize=9)
        ax.set_xlabel("min_samples", fontsize=9)
        ax.set_ylabel("min_cluster_size", fontsize=9)
        ax.set_title(label, fontsize=10, fontweight="bold")

        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                val = pivot.values[i, j]
                if not np.isnan(val):
                    # Highlight target zone
                    border = (metric == "n_clusters" and val < TARGET_MAX_CLUSTERS)
                    ax.text(j, i, f"{val:{fmt}}",
                            ha="center", va="center", fontsize=8,
                            fontweight="bold" if border else "normal",
                            color="black")

        # Garis target
        if metric == "n_clusters":
            ax.set_title(f"{label}  [target < {TARGET_MAX_CLUSTERS} = bold]",
                         fontsize=9, fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "heatmap_grid_search.png", dpi=150, bbox_inches="tight")
plt.show()
print("Tersimpan: heatmap_grid_search.png")


In [ ]:
# ── Line plot: n_clusters vs min_cluster_size per k ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = plt.cm.tab10(np.linspace(0, 1, len(K_NEIGHBORS_GRID)))

for ax, (ymetric, ylabel, hline, hval) in zip(axes, [
    ("n_clusters",  "Jumlah Cluster",   True,  TARGET_MAX_CLUSTERS),
    ("coverage_pct","Coverage (%)",     True,  MIN_COVERAGE),
]):
    for ci, k in enumerate(K_NEIGHBORS_GRID):
        df_k = df_valid[(df_valid["k_neighbors"] == k) &
                        (df_valid["min_samples"] == MIN_SAMPLES_GRID[1])]  # ms tengah
        df_k = df_k.sort_values("min_cluster_size")
        ax.plot(df_k["min_cluster_size"], df_k[ymetric],
                marker="o", label=f"k={k}", color=colors[ci], linewidth=2)

    if hline:
        ax.axhline(hval, color="red", linestyle="--", linewidth=1.5,
                   label=f"Target ({hval})")

    ax.set_xlabel("min_cluster_size", fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(f"{ylabel} vs min_cluster_size  (min_samples={MIN_SAMPLES_GRID[1]})",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "lineplot_grid_search.png", dpi=150, bbox_inches="tight")
plt.show()


## 6. Parameter Terbaik

In [ ]:
# Kriteria: n_clusters < TARGET, coverage >= MIN_COVERAGE, DBCV tertinggi
df_best = df_valid[
    (df_valid["n_clusters"] < TARGET_MAX_CLUSTERS) &
    (df_valid["coverage_pct"] >= MIN_COVERAGE)
].sort_values("dbcv_approx", ascending=False)

print("=" * 60)
print("  PARAMETER TERBAIK — NB04 Hyperparameter Tuning")
print("=" * 60)

if len(df_best) > 0:
    best = df_best.iloc[0]
    print(f"  k_neighbors      : {int(best.k_neighbors)}")
    print(f"  min_cluster_size : {int(best.min_cluster_size)}")
    print(f"  min_samples      : {int(best.min_samples)}")
    print()
    print(f"  n_clusters       : {int(best.n_clusters)}")
    print(f"  coverage         : {best.coverage_pct:.2f}%")
    print(f"  noise            : {best.noise_pct:.2f}%")
    print(f"  DBCV (approx)    : {best.dbcv_approx:.4f}")
    print("=" * 60)
    print()
    print("Gunakan parameter ini di NB02 atau NB05:")
    print(f"  K_NEIGHBORS      = {int(best.k_neighbors)}")
    print(f"  MIN_CLUSTER_SIZE = {int(best.min_cluster_size)}")
    print(f"  MIN_SAMPLES      = {int(best.min_samples)}")

    # Simpan best params
    best_params = {
        "k_neighbors"      : int(best.k_neighbors),
        "min_cluster_size" : int(best.min_cluster_size),
        "min_samples"      : int(best.min_samples),
        "n_clusters"       : int(best.n_clusters),
        "coverage_pct"     : float(best.coverage_pct),
        "noise_pct"        : float(best.noise_pct),
        "dbcv_approx"      : float(best.dbcv_approx),
    }
else:
    best_params = {}
    print("⚠️  Tidak ada kombinasi yang memenuhi kriteria.")
    print(f"   Coba turunkan TARGET_MAX_CLUSTERS atau MIN_COVERAGE.")
    print()
    print("Top 5 terdekat ke target (tanpa filter coverage):")
    fallback = df_valid[df_valid["n_clusters"] < TARGET_MAX_CLUSTERS].sort_values("dbcv_approx", ascending=False)
    print(fallback[cols_show].head(5).to_string(index=False))


## 7. Simpan Hasil

In [ ]:
# Simpan DataFrame lengkap
csv_path = OUTPUT_DIR / "tuning_results.csv"
pkl_path = OUTPUT_DIR / "tuning_results.pkl"

df.to_csv(csv_path, index=False)
with open(pkl_path, "wb") as f:
    pickle.dump({"results_df": df, "best_params": best_params}, f)

print(f"Tersimpan:")
print(f"  {csv_path.name}  ({csv_path.stat().st_size/1024:.1f} KB)")
print(f"  {pkl_path.name}  ({pkl_path.stat().st_size/1024:.1f} KB)")
print()
print(f"Total kombinasi : {len(df)}")
print(f"Valid           : {len(df_valid)}")
print(f"Memenuhi target : {len(df_best)}")
